# Exercice 1 : Mon mot de passe a-t-il fuité ?

> **En une phrase :** les attaquants n'ont pas besoin de deviner vos mots de passe : des milliards sont **déjà publiés**. On vérifie lesquels, **sans jamais envoyer le mot de passe**.

**Le scénario.** En reconnaissance, l'attaquant cherche des « personnes » (Ch. 5) et leurs mots de passe déjà fuités. S'ils sont réutilisés sur le VPN ou le courriel et qu'il n'y a pas de MFA (Ch. 4), il entre sans rien pirater.

**La source.** *Have I Been Pwned – Pwned Passwords* : plus de 900 millions de mots de passe réels, issus de fuites publiques, gratuits et sans inscription.

**Le plan.**
1. Transformer un mot de passe en empreinte (SHA-1).
2. Interroger la base **sans révéler** le mot de passe (k-anonymat).
3. Tester une liste de mots de passe « typiques ».
4. Contre-mesure : refuser les mots de passe fuités à la création.

## Étape 0 : exécutez cette cellule en premier
Le notebook est **autonome** : il fonctionne tel quel sur Colab (ouvert depuis Drive) ou en local. Elle doit afficher **Prêt**.

In [1]:
import hashlib, requests
import pandas as pd
print('Prêt')

Prêt


## Étape 1 : Un mot de passe devient une empreinte
**Ce qu'on fait :** on calcule le SHA-1 du mot de passe. C'est une **empreinte** : toujours la même pour le même mot de passe, et impossible à « défaire ».

In [2]:
mot_de_passe = "motdepasse"          # <-- MODIFIABLE

empreinte = hashlib.sha1(mot_de_passe.encode()).hexdigest().upper()
print("Empreinte SHA-1 :", empreinte)
print("Ce qu'on envoie : ", empreinte[:5], "(seulement les 5 premiers caractères)")

Empreinte SHA-1 : 940C0F26FD5A30775BB1CBD1F6840398D39BB813
Ce qu'on envoie :  940C0 (seulement les 5 premiers caractères)


**👀 Ce qu'on doit voir :** 40 caractères hexadécimaux. On n'enverra que les **5 premiers** au service.

## Étape 2 : Interroger la base sans révéler le mot de passe
**Ce qu'on fait :** on envoie les 5 premiers caractères. Le service renvoie **toutes** les empreintes qui commencent pareil (environ 2 000). C'est **nous** qui cherchons la nôtre dans la liste, localement.

C'est le **k-anonymat** : le service ne sait jamais quel mot de passe on teste.

In [3]:
def nb_fuites(mot_de_passe):
    """Renvoie combien de fois ce mot de passe apparaît dans des fuites publiques."""
    h = hashlib.sha1(mot_de_passe.encode()).hexdigest().upper()
    reponse = requests.get("https://api.pwnedpasswords.com/range/" + h[:5], timeout=10).text
    for ligne in reponse.splitlines():           # chaque ligne : "suite_de_l_empreinte:nombre"
        suite, nombre = ligne.split(":")
        if suite == h[5:]:
            return int(nombre)
    return 0

reponse = requests.get("https://api.pwnedpasswords.com/range/" + empreinte[:5], timeout=10).text
print("Empreintes renvoyées par le service :", len(reponse.splitlines()))
print(f"'{mot_de_passe}' apparaît {nb_fuites(mot_de_passe):,} fois dans des fuites".replace(",", " "))

Empreintes renvoyées par le service : 1950


'motdepasse' apparaît 721 301 fois dans des fuites


**👀 Ce qu'on doit voir :** ≈ 2 000 empreintes renvoyées (la nôtre est cachée dans la foule), et « motdepasse » ≈ **720 000 fois**.

## Étape 3 : Tester des mots de passe « typiques »
**Ce qu'on fait :** on teste des mots de passe qu'un employé de la région pourrait choisir.

In [4]:
# <-- MODIFIABLE : ajoutez vos propres idées (PAS votre vrai mot de passe ici : voir plus bas)
candidats = ["123456", "password", "azerty", "soleil", "motdepasse",
             "Chicoutimi1", "Canadiens1", "Saguenay2024", "Uqac2026!", "J'aimeLeCafé!"]

resultats = pd.DataFrame({"mot de passe": candidats,
                          "nombre de fuites": [nb_fuites(m) for m in candidats]})
resultats.sort_values("nombre de fuites", ascending=False)

,mot de passe,nombre de fuites
0,123456,210461208
1,password,52372427
2,azerty,3355889
3,soleil,838451
4,motdepasse,721301
5,Chicoutimi1,2714
6,Canadiens1,1976
7,Saguenay2024,0
8,Uqac2026!,0
9,J'aimeLeCafé!,0


**👀 Ce qu'on doit voir :** « 123456 » ≈ **210 millions** de fois. Plus surprenant : « Chicoutimi1 » (≈ 2 700) et « Canadiens1 » (≈ 2 000) sont **déjà connus**. « Saguenay2024 », « Uqac2026! » : 0, pour l'instant.

**❓ Question pour la classe :** « Chicoutimi1 » respecte une politique classique (majuscule, chiffre, 11 caractères). Pourtant, il est dans les listes. Pourquoi est-ce grave ?

*Réponse : les attaquants ne devinent pas au hasard, ils essaient d'abord les listes de mots de passe fuités (« credential stuffing »). Un mot de passe « complexe » mais déjà fuité tombe en quelques secondes.*

### Tester votre propre mot de passe (facultatif)
La saisie est masquée et rien n'est affiché ni enregistré. Seuls 5 caractères de l'empreinte quittent votre machine.

In [5]:
# Décommentez les deux lignes suivantes pour tester un de vos mots de passe :
# from getpass import getpass
# print("Nombre de fuites :", nb_fuites(getpass("Mot de passe : ")))

## Étape 4 : Contre-mesure : refuser les mots de passe fuités
**Ce qu'on fait :** une petite fonction de **création de compte** qui applique la recommandation du NIST (SP 800-63B) : refuser tout mot de passe déjà présent dans une fuite.

In [6]:
def accepter_mot_de_passe(m):
    if len(m) < 12:
        return "Refusé : moins de 12 caractères"
    n = nb_fuites(m)
    if n > 0:
        return f"Refusé : déjà vu {n} fois dans des fuites"
    return "Accepté"

for m in ["Chicoutimi1", "motdepasse123", "le-bleuet-du-lac-en-aout"]:
    print(f"{m:28} → {accepter_mot_de_passe(m)}")

Chicoutimi1                  → Refusé : moins de 12 caractères
motdepasse123                → Refusé : déjà vu 39358 fois dans des fuites


le-bleuet-du-lac-en-aout     → Accepté


**👀 Ce qu'on doit voir :** « Chicoutimi1 » refusé (trop court), « motdepasse123 » refusé malgré ses 13 caractères (≈ **39 000 fuites**), et la phrase de passe acceptée.

## Ce qu'il faut retenir
- Des milliards de mots de passe sont **publics**. L'attaquant commence par eux, pas par la force brute.
- On peut vérifier un mot de passe **sans le révéler** (k-anonymat : on n'envoie que 5 caractères de l'empreinte).
- Contre-mesures : refuser les mots de passe fuités, préférer les **phrases de passe**, et surtout le **MFA** (Ch. 4), qui rend le mot de passe fuité inutile à lui seul.

**Exercice suivant :** des personnes aux fichiers. Une simple photo peut dire où vous étiez.